In [ ]:
import torch
import torch.nn as nn

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.block = nn.Sequential(
            nn.Conv2d(in_channels = self.in_channels, 
                      out_channels = self.out_channels,
                      kernel_size = 3, 
                      stride = 1,
                      padding = 1,
                      bias = False),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(inplace = True),


            nn.Conv2d(in_channels = self.out_channels, 
                      out_channels = self.out_channels,
                      kernel_size = 3, 
                      stride = 1,
                      padding = 1,
                      bias = False),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(inplace = True)
        )


    def forward(self, x):

        x = self.block(x)

        return x
        

In [ ]:
class UNet(nn.Module):
    def __init__(self, num_classes, feature_list = [32, 64, 128, 256, 512], bottleneck_size = 1024, image_size = 512):
        super(UNet, self).__init__()

        self.num_classes = num_classes
        self.feature_list = feature_list
        self.bottleneck_size = bottleneck_size
        self.image_size = image_size


        # Model Architecture
        self.encoder = nn.ModuleList()
        self.bottleneck = ConvBlock(
            in_channels = feature_list[-1],
            out_channels = self.bottleneck_size
        )
        self.decoder = nn.ModuleList()
        self.classifier = nn.Conv2d(
            in_channels = feature_list[0],
            out_channels = self.num_classes,
            kernel_size = 1
        )

        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)

        self._make_encoder(self.feature_list)
        self._make_decoder(self.feature_list)
        
        


    def _make_encoder(self, feature_list):

        for feature in self.feature_list:

            encoder_level = ConvBlock(
                in_channels = feature,
                out_channels = feature * 2
            )

            self.encoder.append(encoder_level)
            self.encoder.append(self.pool)

            # self.image_size = self.image_size/2

    
    def _make_decoder(self, feature_list):

        for feature in reversed(self.feature_list):

            upconv = nn.ConvTranspose2d(
                in_channels = feature * 2,
                out_channels = feature,
                kernel_size = 2
            )

            decoder_level = ConvBlock(
                in_channels = feature * 2,
                out_channels = feature
            )

            self.decoder.append(upconv)
            self.decoder.append(decoder_level)


    def forward(self, x):
        
        skips = []

        for i in range(0,  len(self.encoder), 2):       #   Iterate over Encoder Module List, starting from index 0, with a step size of 2 to skip over previous maxpool layer

            x = self.encoder[i](x)      #   Run ConvBlock

            skips.append(x)             #   Save output for skip connection

            x = self.encoder[i + 1](x)  #   Run MaxPool layer  

        x = self.bottleneck(x)

        skips = skips[::-1]   #   Reverse skip connections to start from the last one

        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)

            skip_connection = skips[i//2]   #   Step size of 2 skips over previous upconv layer, this indexing format selects the correct skip conenction

            x = torch.cat((skip_connection, x), dim = 1)

            x = self.decoder[i + 1](x)

        x = self.classifier(x)

        return x



NameError: name 'nn' is not defined